# Pair-Controlled Split Validation

This notebook validates the integrity of the six pair-controlled Puntuguese splits used in the experiments. It checks file availability, record schema, class balance, H/N pair completeness, token-label consistency, expected split sizes, metadata consistency, absence of pair IDs shared across train, validation, and test, and consistency of corpus content across split seeds.


In [1]:
from pathlib import Path
import json
from collections import Counter, defaultdict

import pandas as pd
from IPython.display import display

In [2]:
SPLIT_SEEDS = [13, 21, 40, 42, 73, 101]
SPLIT_NAMES = ["train", "validation", "test"]

EXPECTED_EXAMPLES = {
    "train": 3990,
    "validation": 570,
    "test": 1140,
}

EXPECTED_PAIRS = {
    "train": 1995,
    "validation": 285,
    "test": 570,
}

EXPECTED_CLASS_COUNTS = {
    "train": {0: 1995, 1: 1995},
    "validation": {0: 285, 1: 285},
    "test": {0: 570, 1: 570},
}

REQUIRED_FIELDS = {"id", "text", "label", "tokens", "labels"}

In [3]:
def find_project_root(start_path=None):
    start = Path(start_path or Path.cwd()).resolve()
    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / "data" / "pair_controlled").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing data/pair_controlled."
    )

In [4]:
PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "pair_controlled"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root: {DATA_ROOT}")

Project root: /home/avelar/pair-aware
Data root: /home/avelar/pair-aware/data/pair_controlled


In [5]:
def load_jsonl(path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON in {path} at line {line_number}: {error}"
                ) from error

    return records

In [6]:
def parse_pair_id(record_id):
    if not isinstance(record_id, str):
        raise ValueError("Record ID must be a string.")

    parts = record_id.rsplit(".", 1)

    if len(parts) != 2:
        raise ValueError(f"Invalid record ID format: {record_id}")

    pair_id, suffix = parts

    if not pair_id or suffix not in {"H", "N"}:
        raise ValueError(f"Invalid record ID format: {record_id}")

    return pair_id, suffix

In [7]:
def validate_record(record, split_name, line_index):
    errors = []
    missing_fields = REQUIRED_FIELDS.difference(record)

    if missing_fields:
        errors.append(
            f"{split_name} record {line_index}: missing fields {sorted(missing_fields)}"
        )
        return errors

    try:
        _, suffix = parse_pair_id(record["id"])
    except ValueError as error:
        errors.append(f"{split_name} record {line_index}: {error}")
        return errors

    if not isinstance(record["text"], str) or not record["text"].strip():
        errors.append(f"{split_name} record {line_index}: invalid text")

    if record["label"] not in {0, 1}:
        errors.append(f"{split_name} record {line_index}: invalid label")

    if not isinstance(record["tokens"], list):
        errors.append(f"{split_name} record {line_index}: tokens must be a list")

    if not isinstance(record["labels"], list):
        errors.append(f"{split_name} record {line_index}: labels must be a list")

    if isinstance(record["tokens"], list) and isinstance(record["labels"], list):
        if len(record["tokens"]) != len(record["labels"]):
            errors.append(
                f"{split_name} record {line_index}: token and token-label lengths differ"
            )

        if any(label not in {0, 1} for label in record["labels"]):
            errors.append(
                f"{split_name} record {line_index}: token labels must be binary"
            )

        if suffix == "H" and not any(record["labels"]):
            errors.append(
                f"{split_name} record {line_index}: pun instance has no positive token label"
            )

        if suffix == "N" and any(record["labels"]):
            errors.append(
                f"{split_name} record {line_index}: non-pun instance has positive token labels"
            )

    expected_label = 1 if suffix == "H" else 0

    if record["label"] != expected_label:
        errors.append(
            f"{split_name} record {line_index}: suffix and instance label are inconsistent"
        )

    return errors

In [8]:
def validate_partition(records, split_name):
    errors = []
    ids = [record.get("id") for record in records]
    duplicate_ids = [record_id for record_id, count in Counter(ids).items() if count > 1]

    for index, record in enumerate(records, start=1):
        errors.extend(validate_record(record, split_name, index))

    if duplicate_ids:
        errors.append(
            f"{split_name}: duplicate record IDs found: {duplicate_ids[:10]}"
        )

    pair_members = defaultdict(list)

    for record in records:
        try:
            pair_id, suffix = parse_pair_id(record.get("id"))
            pair_members[pair_id].append(suffix)
        except ValueError:
            continue

    invalid_pairs = {
        pair_id: suffixes
        for pair_id, suffixes in pair_members.items()
        if sorted(suffixes) != ["H", "N"]
    }

    if invalid_pairs:
        sample = list(invalid_pairs.items())[:10]
        errors.append(f"{split_name}: incomplete or invalid H/N pairs found: {sample}")

    class_counts = Counter(
        record.get("label") for record in records if record.get("label") in {0, 1}
    )

    if len(records) != EXPECTED_EXAMPLES[split_name]:
        errors.append(
            f"{split_name}: expected {EXPECTED_EXAMPLES[split_name]} examples, found {len(records)}"
        )

    if len(pair_members) != EXPECTED_PAIRS[split_name]:
        errors.append(
            f"{split_name}: expected {EXPECTED_PAIRS[split_name]} pairs, found {len(pair_members)}"
        )

    if dict(class_counts) != EXPECTED_CLASS_COUNTS[split_name]:
        errors.append(
            f"{split_name}: expected class counts {EXPECTED_CLASS_COUNTS[split_name]}, found {dict(class_counts)}"
        )

    return {
        "records": records,
        "ids": set(ids),
        "pair_ids": set(pair_members),
        "class_counts": dict(class_counts),
        "errors": errors,
    }

In [9]:
def validate_metadata(seed_dir, computed):
    errors = []
    metadata_path = seed_dir / "metadata.json"

    if not metadata_path.is_file():
        return ["metadata.json is missing"]

    with metadata_path.open("r", encoding="utf-8") as file:
        metadata = json.load(file)

    expected_values = {
        "strategy": "pair_controlled",
        "seed": computed["seed"],
        "total_examples": computed["total_examples"],
        "total_pairs": computed["total_pairs"],
        "train_examples": computed["train_examples"],
        "validation_examples": computed["validation_examples"],
        "test_examples": computed["test_examples"],
        "cross_split_pairs": computed["cross_split_pairs"],
        "cross_split_rate": 0.0,
    }

    for key, expected in expected_values.items():
        if metadata.get(key) != expected:
            errors.append(
                f"metadata.json: {key} expected {expected}, found {metadata.get(key)}"
            )

    for split_name in SPLIT_NAMES:
        metadata_counts = metadata.get("class_distribution", {}).get(split_name, {})
        normalized_counts = {
            int(label): count for label, count in metadata_counts.items()
        }

        if normalized_counts != computed[f"{split_name}_class_counts"]:
            errors.append(
                f"metadata.json: class distribution mismatch for {split_name}"
            )

    return errors

In [10]:
def validate_seed(seed):
    seed_dir = DATA_ROOT / f"seed_{seed}"
    errors = []

    if not seed_dir.is_dir():
        return {
            "seed": seed,
            "valid": False,
            "errors": [f"Missing directory: {seed_dir}"],
        }

    partitions = {}

    for split_name in SPLIT_NAMES:
        split_path = seed_dir / f"{split_name}.jsonl"

        if not split_path.is_file():
            errors.append(f"Missing file: {split_path}")
            continue

        records = load_jsonl(split_path)
        partitions[split_name] = validate_partition(records, split_name)
        errors.extend(partitions[split_name]["errors"])

    if len(partitions) != len(SPLIT_NAMES):
        return {
            "seed": seed,
            "valid": False,
            "errors": errors,
        }

    train_pairs = partitions["train"]["pair_ids"]
    validation_pairs = partitions["validation"]["pair_ids"]
    test_pairs = partitions["test"]["pair_ids"]

    crossing_pairs = (
        train_pairs.intersection(validation_pairs)
        | train_pairs.intersection(test_pairs)
        | validation_pairs.intersection(test_pairs)
    )

    if crossing_pairs:
        errors.append(
            f"Pair IDs shared across partitions: {sorted(crossing_pairs)[:10]}"
        )

    all_ids = set().union(*(partitions[name]["ids"] for name in SPLIT_NAMES))
    all_pair_ids = set().union(
        *(partitions[name]["pair_ids"] for name in SPLIT_NAMES)
    )

    total_examples = sum(
        len(partitions[name]["records"]) for name in SPLIT_NAMES
    )
    total_pairs = len(all_pair_ids)

    if len(all_ids) != total_examples:
        errors.append("Duplicate record IDs exist across partitions")

    if total_examples != 5700:
        errors.append(f"Expected 5700 total examples, found {total_examples}")

    if total_pairs != 2850:
        errors.append(f"Expected 2850 total pairs, found {total_pairs}")

    computed = {
        "seed": seed,
        "total_examples": total_examples,
        "total_pairs": total_pairs,
        "train_examples": len(partitions["train"]["records"]),
        "validation_examples": len(partitions["validation"]["records"]),
        "test_examples": len(partitions["test"]["records"]),
        "train_pairs": len(train_pairs),
        "validation_pairs": len(validation_pairs),
        "test_pairs": len(test_pairs),
        "train_class_counts": partitions["train"]["class_counts"],
        "validation_class_counts": partitions["validation"]["class_counts"],
        "test_class_counts": partitions["test"]["class_counts"],
        "cross_split_pairs": len(crossing_pairs),
    }

    metadata_errors = validate_metadata(seed_dir, computed)
    errors.extend(metadata_errors)

    return {
        **computed,
        "all_ids": all_ids,
        "all_pair_ids": all_pair_ids,
        "records_by_split": {
            name: partitions[name]["records"] for name in SPLIT_NAMES
        },
        "valid": len(errors) == 0,
        "errors": errors,
    }

In [11]:
def build_record_signatures(seed_result):
    signatures = {}

    for split_name, records in seed_result["records_by_split"].items():
        for record in records:
            signatures[record["id"]] = (
                record["text"],
                record["label"],
                tuple(record["tokens"]),
                tuple(record["labels"]),
            )

    return signatures

In [12]:
def validate_cross_seed_consistency(seed_results):
    errors = []
    reference = seed_results[0]
    reference_ids = reference["all_ids"]
    reference_pair_ids = reference["all_pair_ids"]
    reference_signatures = build_record_signatures(reference)

    for result in seed_results[1:]:
        if result["all_ids"] != reference_ids:
            errors.append(
                f"Seed {result['seed']} does not contain the same record IDs as seed {reference['seed']}"
            )

        if result["all_pair_ids"] != reference_pair_ids:
            errors.append(
                f"Seed {result['seed']} does not contain the same pair IDs as seed {reference['seed']}"
            )

        signatures = build_record_signatures(result)

        if signatures != reference_signatures:
            errors.append(
                f"Seed {result['seed']} contains record content inconsistent with seed {reference['seed']}"
            )

    return errors

In [13]:
seed_results = [validate_seed(seed) for seed in SPLIT_SEEDS]

summary_rows = []

for result in seed_results:
    summary_rows.append(
        {
            "seed": result["seed"],
            "valid": result["valid"],
            "train_examples": result.get("train_examples"),
            "validation_examples": result.get("validation_examples"),
            "test_examples": result.get("test_examples"),
            "train_pairs": result.get("train_pairs"),
            "validation_pairs": result.get("validation_pairs"),
            "test_pairs": result.get("test_pairs"),
            "train_non_pun": result.get("train_class_counts", {}).get(0),
            "train_pun": result.get("train_class_counts", {}).get(1),
            "validation_non_pun": result.get("validation_class_counts", {}).get(0),
            "validation_pun": result.get("validation_class_counts", {}).get(1),
            "test_non_pun": result.get("test_class_counts", {}).get(0),
            "test_pun": result.get("test_class_counts", {}).get(1),
            "cross_split_pairs": result.get("cross_split_pairs"),
            "error_count": len(result["errors"]),
        }
    )

validation_summary = pd.DataFrame(summary_rows)
display(validation_summary)

,seed,valid,train_examples,validation_examples,test_examples,train_pairs,validation_pairs,test_pairs,train_non_pun,train_pun,validation_non_pun,validation_pun,test_non_pun,test_pun,cross_split_pairs,error_count
0,13,True,3990,570,1140,1995,285,570,1995,1995,285,285,570,570,0,0
1,21,True,3990,570,1140,1995,285,570,1995,1995,285,285,570,570,0,0
2,40,True,3990,570,1140,1995,285,570,1995,1995,285,285,570,570,0,0
3,42,True,3990,570,1140,1995,285,570,1995,1995,285,285,570,570,0,0
4,73,True,3990,570,1140,1995,285,570,1995,1995,285,285,570,570,0,0
5,101,True,3990,570,1140,1995,285,570,1995,1995,285,285,570,570,0,0


In [14]:
cross_seed_errors = validate_cross_seed_consistency(seed_results)

all_errors = {
    result["seed"]: result["errors"]
    for result in seed_results
    if result["errors"]
}

if cross_seed_errors:
    all_errors["cross_seed"] = cross_seed_errors

if all_errors:
    for key, errors in all_errors.items():
        print(f"Validation errors for {key}:")
        for error in errors:
            print(f"  {error}")
    raise AssertionError("Pair-controlled split validation failed.")

print("All pair-controlled splits passed validation.")
print("Each seed contains 5,700 examples and 2,850 complete H/N pairs.")
print("No pair ID crosses train, validation, and test partitions.")
print("All six seeds contain the same corpus records and annotations.")

All pair-controlled splits passed validation.
Each seed contains 5,700 examples and 2,850 complete H/N pairs.
No pair ID crosses train, validation, and test partitions.
All six seeds contain the same corpus records and annotations.
